# Exercise 2b: Feature engineering

In [3]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import re
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
X_train = pd.read_csv("ex2_train.csv")
y_train = pd.read_csv("ex2_class_train.csv")
X_test = pd.read_csv("ex2_test.csv")
y_test = pd.read_csv("ex2_class_test.csv")

In [5]:
# define a utility function to print out the prediction performance
def evaluate_result(y_test, y_pred, clf):
    print(f'Accuracy: {accuracy_score(y_test, y_pred):.4f}')
    print(f'Precision: {precision_score(y_test, y_pred):.4f}')
    print(f'Recall: {recall_score(y_test, y_pred):.4f}')
    print(f'F1-score: {f1_score(y_test, y_pred):.4f}')
    print(f'AUC-ROC: {roc_auc_score(y_test, clf.predict_proba(X_test_processed)[:, 1]):.4f}')

## Prototyping (without feature engineering)

In [6]:
def preprocess(data_in):
    data = data_in.drop(columns=['Name'])
    
    data = data.fillna({
        'Age': data['Age'].median(),
        'Embarked': data['Embarked'].mode(dropna=True).iloc[0],
        'Fare': data['Fare'].median()
    })

    # Convert categorical variables to dummy/indicator variables
    data = pd.get_dummies(data, columns=['Sex', 'Embarked'], drop_first=True)

    return data

In [7]:
X_train_processed = preprocess(X_train)
X_test_processed = preprocess(X_test)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_processed, y_train.values.ravel())
y_pred = clf.predict(X_test_processed)

print('Random Forest Model without Feature Engineering')
evaluate_result(y_test, y_pred, clf)

Random Forest Model without Feature Engineering
Accuracy: 0.8101
Precision: 0.7778
Recall: 0.7568
F1-score: 0.7671
AUC-ROC: 0.8736


## Feature engineering

The classification using simple preprocessed data gives only mediocre performance.

**TODO: You should make use of the insights from your EDA (ex2a) to complete the following feature engineering function below.** Later the function will replace the simple preprocessing.

You will pass the exercise if your feature engineering can improve the performance (i.e., winning in three or more metrics).

In [8]:
def feature_engineering(data_in):
    df = data_in.copy()

    df['Title'] = df['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
    df['Title'] = df['Title'].replace({'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs'})
    common_titles = ['Mr', 'Mrs', 'Miss', 'Master']
    df['Title'] = df['Title'].where(df['Title'].isin(common_titles), 'Rare')

    df = df.drop(columns=['Name'])
    df = df.fillna({
        'Age': df['Age'].median(),
        'Embarked': df['Embarked'].mode(dropna=True).iloc[0],
        'Fare': df['Fare'].median()
    })
    
    df['is_child'] = (df['Age'] <= 10).astype(int)
    df['first_class'] = (df['Pclass'] == 1).astype(int)
    df['second_class'] = (df['Pclass'] == 2).astype(int)
    df['third_class'] = (df['Pclass'] == 3).astype(int)

    df['child_first_class'] = df['is_child'] * df['first_class']
    df['child_second_class'] = df['is_child'] * df['second_class']
    df['child_third_class'] = df['is_child'] * df['third_class']

    df['family_size'] = df['SibSp'] + df['Parch'] + 1 
    df['is_alone'] = (df['family_size'] == 1).astype(int)
    df['large_family'] = (df['family_size'] > 4).astype(int)
    
    df = pd.get_dummies(df, columns=['Sex', 'Embarked', 'Title'], drop_first=True)

    return df

In [10]:
X_train_processed = feature_engineering(X_train)
X_test_processed = feature_engineering(X_test)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train_processed, y_train.values.ravel())
y_pred = clf.predict(X_test_processed)

print('Random Forest Model with Feature Engineering')
evaluate_result(y_test, y_pred, clf)

Random Forest Model with Feature Engineering
Accuracy: 0.8212
Precision: 0.7763
Recall: 0.7973
F1-score: 0.7867
AUC-ROC: 0.8825
